   ... expected to know how to use CBuild/`cb`, plus normal C tools like `gcc`,
   `gdb` and `valgrind`. The PDF also says compilation/crashing penalties can
   be harsh, so clean compiling code matters a lot.

   Source context...

---
KEY CONCEPT
   Raw compiler command:
   
```c
gcc -std=c17 -Wall -Wpedantic -o prog prog.c
```
   means: manually tell GCC flags, output names, and source files.

   CBuild/`cb` is closer to:

```c
cb
```
   meaning: read the provided build config, work out what to compile/link, then
   call the compiler for you.

   So mentally:
```
gcc/clang = direct compiler command
make      = build automation using Makefile rules
cb        = course build automation using provided .cbuild/.build config
```
   You do not need to write pattern rules. You mainly need to know:
```bash
cb
cb clean
```
   and understand that GCC warnings/errors still matter because `cb` eventually
   invokes GCC/Clang underneath. 


---

   Use the `wc` command to count the number of lines, words, and bytes in the 
   files specified by the File parameter. If a file is not specified for the
   File parameter, standard input is used.

```sh
cb
cb --clean
cb --allclean
cb --test
cb --install
```

   Locally `cb` is not on this machine's PATH... 

---
CONCEPTS
   CBuild/`cb` is basically "Course Make without writing Makefiles."

   In `08.cbuild/test1/.cbuild`, the whole config is:
```make
BUILD = avgwordlen testlist
```
   That means: by default, build two executable programs, `avgwordlen` and
   `testlist`.

   `cb` scans the ``.c` and `.h` files, especially local includes like:

```c
#include "intlist.h"
```
   Then it infers dependencies:

```
testlist.c includes intlist.h
intlist.h has matching intlist.c
therefore testlist needs testlist.o + intlist.o
```
   The raw GCC equivalent would be:
```sh
gcc -Wall -c intlist.c
gcc -Wall -c testlist.c
gcc testlist.o intlist.o -o testlist
```

   ... But wth `cb`, you just do:
```sh
cb
```

   Important distinction:
```
compiler error = bad C syntax/types in one source file
linker error   = object files compiled, but final executable cannot be connected
runtime crash  = program built, then segfaulted/asserted/etc
```

...
```c
intlist_append(xs, 3);
```
   Linker error if `intlist_append` is declared in `.h` but not implemented in 
   `.c`.
```
int *p = NULL;
*p = 3;
```
   Runtime crash..

   For clean builds:
```sh
cb --clean
```
   removes generated objects/executables.

```sh
cb --allclean
```
   cleans and rebuilds.

   This matters in exams because stale object files can hide problems, and the
   hints PDF says non-compiling/crashing tasks get heavy penalties.

---

Q1

```sh
cb
```

---
Q2

   Conceptual hint: `avgwordlen.c` and `testlist.c` are both main programs.

   Syntax hints: multiple `main` functions cannot be linked into one executable.

   ANSWER: It tries to compile/link every `.c` file into one program. If more
   than one file has `main`, linking fails with a multiple-definition error.


---
Q3

   Manually, what two-stage GCC process builds `testlist` from `testlist.c`
   and `intlist.c`?

   Conceptual hint: first compile `.c -> .o`, then link `.o -> executable`.

   Syntax hint: use `-c` for compile-only.

```sh
gcc -Wall -c testlist.c
gcc -Wall -Wextra -c intlist.c
gcc testlist.o intlist.o -o testlist
```

---
Q4
   `testlist.c` includes `intlist.h`. There is also an `intlist.c`. What does
   `cb` infer?

   CONCEPTUAL HINT: matching `.h` and `.c` form a module.

   Answer: `testlist` depends on the `intlist` module, so `cb` should compile 
   and link `intlist.o` into `testlist`.

---
Q5
   ... implementation changed, not interface.

   ANSWER: `intlist.o` should recompile, and programs using it, such as 
   `testlist` and `avgwordlen`, should relink.

---
Q6
   ... interface changed. ... Anything including `intlist.h` should recompile,
   so likely `intlist.o`, `testlist.o`, `avgwordlen.o`, then the executables
   relink.

---
Q7
```sh
undefined reference to `intlist_length`
```
   ... This is link error...

   ANSWER: Linker error. The function was declared or called, but no matching
   implementation was found during linking. 

---
Q8
```
warning: implicit declaration of function `foo`
```
   CONCEPTUAL HINT: Compiler has not seen the function prototype.

   ANSWER: Missing header include, misspelled function name, or no declaration
   before use. In C17 with strict flags, treat this as serious.

   warning: implicit declaration of function `foo`

---
Q9
   In current `cb` syntax, what command forces a clean rebuild?

```sh
cb --allclean
```

---
Q10
   ... Your solution to one task has broken helper function that stops 
   compilation. What should you do.

   ANSWER: Comment out or isolate the broken code so the task compiles. A 
   smaller compiling solution is better than a larger non-compiling one. 

---

   This specific compiler error normally happens when a developer is 
   transitioning from small, single-file hobby scripts to MULTI-FILE SOFTWARE
   ARCHITECTURES. In a simple project, running a single command like 
   `gcc main.c -o program` works because the compiler handles both compilation
   and linking in one seamless pass. However, when a second like `intlist.c` is 
   introduced, the natural but mistaken instinct is to just copy-paste the new
   filename into the middle of the old command and 

---

   In C, a string is not a first-class object type; it is simply a contiguous
   array of characters in memory that concludes with a special marker called the
   NULL TERMINATOR (`\0`). Because there are no built-in properties like `.len`
   or `.length`, finding the length of a string requires scanning through
   memory byte-by-byte until this null terminator is encountered.

1. The Standard Way: `strlen()`
   The canonical method to find the length of a `const char*` is using the 
   `strlen()` function provided by the standard library header `<string.h>`.

```c
#include <stdio.h>
#include <string.h>

int main() {
    const char *str = "ARMv8";

    // strlen returns a size_t (unsigned integer type)
    size_t len = strlen(str);

    printf("The length is: %zu\n", len);    // Outputs: 5
    return 0;
}
```
   - EXCLUDES THE NULL TERMINATOR: `strlen()` counts only the readable 
     characters. For `"ARMv8"`, it returns `5`, even though the array occupies
     `6` bytes of memory in total to store the trailing `\0`.
   - TIME COMPLEXITY: This operation is O(N). Because C strings do not store
     their length complexity. `strlen()` must inspect every single character
     sequentially. Calling `strlen()` inside the condition of a loop 
     (e.g., `for (size_t i = 0; i < strlen(s); i++)`) creates an accidental
     O(N^2) performance bottleneck.


---
2. THE DEFENSIVE WAY: `strnlen()`
   If you are dealing with untrusted input or raw binary packets where a string 
   might be malformed or missing its null terminator, `strlen()` will read
   past the buffer boundaries until it crashes with a segmentation fault.

   To prevent this, use `strnlen()`. It requires a secondary argument specifying
   the maximum number of bytes to examine. 

```c
#include <string.h>

const char *uncrusted_buffer = get_network_packet();
// Stops scanning at the null terminator OR after 100 bytes, whichever comes
// first
size_t len = strnlen(untrusted_buffer, 100);
```


---
3. THE MANUAL WAY (Under the Hood)
   To understand exactly how the machine calculates this value without library
   functions, you can implemnent the counting logic manually using a pointer or 
   an index tracker:

```c
size_t manual_strlen(const char *s) {
    if (s == NULL) {
        return NULL;
    }

    size_t length = 0;
    while (s[length] != '\0') {
        length++;
    }
    return length;
}
```

   - `strlen(ptr)` computes the number of characters up to the null terminator
     by evaluating the memory contents at runtime.
   - `sizeof(ptr)` computes the size of the pointer variable itself at compile
     time. On a 64-bit architecture, a pointer is always 8 bytes, regardless of
     whether the string it points to contains 2 characters or 20,000 characters.


---

---